# Image only Regression Model
This part implements image only regression models using Swin backbone architecture. The model extracts visual features from input images and predicts land cover composition percentages through a regression head. This serves as a baseline to evaluate how well visual information alone can capture area distribution without any additional semantic input.

In [1]:
import os
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import torchvision
from torchvision import transforms, models

from sklearn.metrics import mean_absolute_error, mean_squared_error

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
DATA_ROOT = "/content/drive/MyDrive/subset"

In [4]:
import os

print(os.listdir(DATA_ROOT))

['captions_subset.csv', '.DS_Store', 'images']


In [5]:
DATA_ROOT = "/content/drive/MyDrive/subset"
CSV_PATH = os.path.join(DATA_ROOT, "captions_subset.csv")
IMAGE_DIR = os.path.join(DATA_ROOT, "images")

TARGET_COLS = ["Tree", "Shrub", "Grass", "Crop", "Built-up", "Barren", "Water"]

BATCH_SIZE = 32
IMG_SIZE = 224
EPOCHS = 8
LR = 1e-4

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

Device: cuda


In [6]:
df = pd.read_csv(CSV_PATH)

print(df.head())
print("Dataset size:", len(df))

     filename  split  Tree  Shrub  Grass  Crop  Built-up  Barren  Water  \
0  267687.png  synth    85     13      2     0         0       0      0   
1  222448.png  synth    18      0     81     0         0       1      0   
2   14328.png  synth     2      0     65     1        22      10      0   
3  224249.png  synth    61      0     39     0         0       0      0   
4  218094.png  synth     0      0      5    93         2       0      0   

                                    hybrid_gemma3-4b  \
0  The image depicts a heavily forested mountaino...   
1  The image depicts a landscape dominated by ext...   
2  The image depicts a landscape dominated by ext...   
3  The image depicts a mountainous region dominat...   
4  The image depicts a landscape dominated by ext...   

                                  hybrid_qwen3-vl-8b  \
0  The scene is dominated by dense tree cover (85...   
1  The scene depicts a predominantly grass-covere...   
2  The scene is dominated by grasslands (65%

In [7]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)

print("Train:", len(train_df))
print("Val:", len(val_df))

Train: 400
Val: 100


In [8]:
class AreaDataset(Dataset):
    def __init__(self, df, image_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img_path = os.path.join(self.image_dir, row["filename"])
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        target = torch.tensor(row[TARGET_COLS].values.astype(np.float32))

        return image, target

In [9]:
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

train_ds = AreaDataset(train_df, IMAGE_DIR, train_transform)
val_ds   = AreaDataset(val_df, IMAGE_DIR, val_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

For the Swin image-only model, input images are resized to 224×224, converted to tensors, and normalized using standard ImageNet mean and standard deviation values. During training, a random horizontal flip is applied as a lightweight augmentation to improve generalization, while validation images are only resized and normalized. This preprocessing is consistent with the expectations of the pretrained Swin-T backbone and supports transfer learning.

# SWIN Transformer for Regression

In [10]:
class SwinImageRegressor(nn.Module):
    def __init__(self, num_outputs=7):
        super().__init__()

        self.backbone = models.swin_t(weights=models.Swin_T_Weights.DEFAULT)
        in_features = self.backbone.head.in_features
        self.backbone.head = nn.Linear(in_features, num_outputs)

    def forward(self, x):
        logits = self.backbone(x)
        preds = torch.softmax(logits, dim=1) * 100.0
        return preds

In [11]:
model = SwinImageRegressor(num_outputs=len(TARGET_COLS)).to(DEVICE)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

Downloading: "https://download.pytorch.org/models/swin_t-704ceda3.pth" to /root/.cache/torch/hub/checkpoints/swin_t-704ceda3.pth


100%|██████████| 108M/108M [00:00<00:00, 217MB/s]


In [12]:
from tqdm import tqdm
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error

def train_epoch_image_only():
    model.train()
    total_loss = 0.0

    loop = tqdm(train_loader, desc="Training")

    for imgs, targets in loop:
        imgs = imgs.to(DEVICE)
        targets = targets.to(DEVICE)

        optimizer.zero_grad()
        preds = model(imgs)
        loss = criterion(preds, targets)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * imgs.size(0)
        loop.set_postfix(loss=loss.item())

    return total_loss / len(train_loader.dataset)


def evaluate_image_only():
    model.eval()
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for imgs, targets in tqdm(val_loader, desc="Validation"):
            imgs = imgs.to(DEVICE)
            preds = model(imgs).cpu().numpy()

            all_preds.append(preds)
            all_targets.append(targets.numpy())

    y_pred = np.vstack(all_preds)
    y_true = np.vstack(all_targets)

    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    per_class_mae = np.mean(np.abs(y_true - y_pred), axis=0)

    return mae, rmse, per_class_mae, y_true, y_pred

In [13]:
best_mae = float("inf")
best_state = None

for epoch in range(EPOCHS):
    loss = train_epoch_image_only()
    mae, rmse, _, _, _ = evaluate_image_only()

    print(f"Epoch {epoch+1}")
    print(f"  Loss: {loss:.4f}")
    print(f"  MAE:  {mae:.4f}")
    print(f"  RMSE: {rmse:.4f}")

    if mae < best_mae:
        best_mae = mae
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        print("  --> saved best model")

Validation: 100%|██████████| 4/4 [00:24<00:00,  6.05s/it]


Epoch 1
  Loss: 345.9344
  MAE:  6.9354
  RMSE: 13.9409
  --> saved best model


Validation: 100%|██████████| 4/4 [00:00<00:00,  4.57it/s]


Epoch 2
  Loss: 157.3439
  MAE:  5.6135
  RMSE: 11.4943
  --> saved best model


Validation: 100%|██████████| 4/4 [00:00<00:00,  4.78it/s]


...

Epoch 6
  Loss: 48.6324
  MAE:  4.0645
  RMSE: 8.0368
  --> saved best model


Validation: 100%|██████████| 4/4 [00:00<00:00,  4.49it/s]


Epoch 7
  Loss: 38.8825
  MAE:  3.9621
  RMSE: 8.0241
  --> saved best model


Validation: 100%|██████████| 4/4 [00:00<00:00,  4.47it/s]

Epoch 8
  Loss: 47.1215
  MAE:  3.8655
  RMSE: 7.3098
  --> saved best model


In [14]:
model.load_state_dict(best_state)

mae, rmse, per_class_mae, y_true, y_pred = evaluate_image_only()

print("Best Val MAE:", mae)
print("Best Val RMSE:", rmse)

for col, score in zip(TARGET_COLS, per_class_mae):
    print(f"{col}: {score:.2f}")

Validation: 100%|██████████| 4/4 [00:00<00:00,  4.58it/s]

Best Val MAE: 3.865527391433716
Best Val RMSE: 7.309809523995794
Tree: 6.08
Shrub: 1.28
Grass: 8.77
Crop: 5.97
Built-up: 1.09
Barren: 2.62
Water: 1.25


#  Text and SWIN Multimodel for regression
This section extends the baseline by incorporating textual descriptions alongside images. Image features are combined with text embeddings obtained from a pretrained language model, and the fused representation is used for regression. This setup enables the model to leverage both visual and semantic information, allowing evaluation of the impact of multimodal learning on area estimation performance.

In [15]:
class SwinTextConcatRegressor(nn.Module):
    def __init__(self, num_outputs=7):
        super().__init__()

        # Image encoder: Swin Transformer
        self.image_backbone = models.swin_t(weights=models.Swin_T_Weights.DEFAULT)
        img_dim = self.image_backbone.head.in_features
        self.image_backbone.head = nn.Identity()
        self.image_proj = nn.Linear(img_dim, 256)

        # Text encoder: DistilBERT
        self.text_backbone = AutoModel.from_pretrained("distilbert-base-uncased")
        text_dim = self.text_backbone.config.hidden_size
        self.text_proj = nn.Linear(text_dim, 256)

        # Fusion head
        self.regressor = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, num_outputs)
        )

    def forward(self, image, input_ids, attention_mask):
        # Image
        img_feat = self.image_backbone(image)
        img_feat = self.image_proj(img_feat)

        # Text
        text_out = self.text_backbone(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        text_feat = text_out.last_hidden_state[:, 0, :]
        text_feat = self.text_proj(text_feat)

        # Concat fusion
        fused = torch.cat([img_feat, text_feat], dim=1)

        logits = self.regressor(fused)
        preds = torch.softmax(logits, dim=1) * 100.0
        return preds

In [16]:
class MultiModalAreaDataset(Dataset):
    def __init__(self, df, image_dir, text_col, target_cols, tokenizer, max_len=128, transform=None):
        self.df = df.reset_index(drop=True)
        self.image_dir = image_dir
        self.text_col = text_col
        self.target_cols = target_cols
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img_path = os.path.join(self.image_dir, row["filename"])
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)

        text = str(row[self.text_col])
        enc = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )

        input_ids = enc["input_ids"].squeeze(0)
        attention_mask = enc["attention_mask"].squeeze(0)

        target = torch.tensor(row[self.target_cols].values.astype(np.float32), dtype=torch.float32)

        return {
            "image": image,
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "target": target
        }

In [17]:
from transformers import AutoTokenizer

TEXT_COL = "vision_qwen3-vl-8b"
MAX_LEN = 128

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Among the available caption sources, the vision generated caption was used because it provides a natural textual summary of the image content and is  aligned with the visual input. This does not constitute data leakage, because the caption is derived only from the image itself and does not use the ground truth area percentages or segmentation labels as input. In other words, the model receives additional semantic information extracted from the same image, but not target information.

In [18]:
train_ds = MultiModalAreaDataset(
    train_df, IMAGE_DIR, TEXT_COL, TARGET_COLS, tokenizer, max_len=MAX_LEN, transform=train_transform
)

val_ds = MultiModalAreaDataset(
    val_df, IMAGE_DIR, TEXT_COL, TARGET_COLS, tokenizer, max_len=MAX_LEN, transform=val_transform
)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

In the multimodal Swin setup, the same image preprocessing pipeline is used as in the image only model. In addition, textual descriptions are tokenized using a pretrained DistilBERT tokenizer, with a fixed maximum sequence length. The Swin image features and DistilBERT text features are projected to a common space and concatenated before the regression head. This allows the model to use both visual and semantic information to estimate class composition percentages.

In [19]:
import os
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from torchvision import transforms, models
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error

from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm

In [20]:
model = SwinTextConcatRegressor(num_outputs=len(TARGET_COLS)).to(DEVICE)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [23]:
from tqdm import tqdm
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error

def train_epoch_multimodal():
    model.train()
    total_loss = 0.0

    loop = tqdm(train_loader, desc="Training")

    for batch in loop:
        images = batch["image"].to(DEVICE)
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        targets = batch["target"].to(DEVICE)

        optimizer.zero_grad()
        preds = model(images, input_ids, attention_mask)
        loss = criterion(preds, targets)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)
        loop.set_postfix(loss=loss.item())

    return total_loss / len(train_loader.dataset)


def evaluate_multimodal():
    model.eval()
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Validation"):
            images = batch["image"].to(DEVICE)
            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            targets = batch["target"]

            preds = model(images, input_ids, attention_mask).cpu().numpy()

            all_preds.append(preds)
            all_targets.append(targets.numpy())

    y_pred = np.vstack(all_preds)
    y_true = np.vstack(all_targets)

    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    per_class_mae = np.mean(np.abs(y_true - y_pred), axis=0)

    return mae, rmse, per_class_mae, y_true, y_pred

In [24]:
best_mae = float("inf")
best_state = None

for epoch in range(EPOCHS):
    loss = train_epoch_multimodal()
    mae, rmse, _, _, _ = evaluate_multimodal()

    print(f"Epoch {epoch+1}")
    print(f"  Loss: {loss:.4f}")
    print(f"  MAE:  {mae:.4f}")
    print(f"  RMSE: {rmse:.4f}")

    if mae < best_mae:
        best_mae = mae
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        print("  --> saved best model")

Validation: 100%|██████████| 4/4 [00:00<00:00,  4.08it/s]


Epoch 1
  Loss: 54.6418
  MAE:  3.8802
  RMSE: 8.0156
  --> saved best model


Validation: 100%|██████████| 4/4 [00:01<00:00,  3.82it/s]


Epoch 2
  Loss: 46.6147
  MAE:  4.3282
  RMSE: 9.1131


Validation: 100%|██████████| 4/4 [00:01<00:00,  3.92it/s]


...

Epoch 6
  Loss: 33.1793
  MAE:  3.7893
  RMSE: 7.7369


Validation: 100%|██████████| 4/4 [00:00<00:00,  4.02it/s]


Epoch 7
  Loss: 32.9078
  MAE:  3.6981
  RMSE: 8.0441


Validation: 100%|██████████| 4/4 [00:01<00:00,  3.93it/s]

Epoch 8
  Loss: 24.0232
  MAE:  3.7034
  RMSE: 7.4778


In [25]:
model.load_state_dict(best_state)

mae, rmse, per_class_mae, y_true, y_pred = evaluate_multimodal()

print("Best Val MAE:", mae)
print("Best Val RMSE:", rmse)

for col, score in zip(TARGET_COLS, per_class_mae):
    print(f"{col}: {score:.2f}")

Validation: 100%|██████████| 4/4 [00:01<00:00,  3.90it/s]

Best Val MAE: 3.691579580307007
Best Val RMSE: 7.462053560995657
Tree: 4.66
Shrub: 1.23
Grass: 8.68
Crop: 6.41
Built-up: 1.10
Barren: 2.71
Water: 1.04


The Swin Transformer models show a smaller but still meaningful improvement when incorporating textual information. The image only model achieves a validation MAE of 3.87 and RMSE of 7.31, while the multimodal model slightly improves to 3.69 MAE and 7.46 RMSE. Although the MAE decreases, the RMSE remains similar, indicating modest but consistent gains from text integration.

At the class level, improvements are observed in several categories. Tree (MAE 6.08 to 4.66) and Grass (MAE 8.77 to 8.68) show reductions in error, suggesting that text helps refine predictions for dominant vegetation classes. Water  and Barrenemain relatively stable.

Overall, compared to DeiT, the Swin model already achieves strong performance in the image only setting, and therefore gains from text are more limited. This supports that stronger visual feature extractors may rely less on additional textual information.